# ClientCo — Périmètre du modèle : churn total ou partiel, quels segments ?

**Objectif** : avant de modéliser, combien de CA est **en jeu** selon la définition de la perte (total, partiel, durable, interruption) et selon le segment ciblé ? C'est le plafond de CA, première moitié de l'arbitrage ; la fiabilité du modèle par label est dans `05_label_test_and_list_allocation`. Aucun modèle ici. À la fin : le périmètre des réguliers recalculé sur **180 jours** (celui qu'utilisent les modèles).

Jour d'achat = CA net des retours du jour > 0,01 €. Dates d'origine du dataset.

## 0. Setup

Le CSV fait **4,2 Go / 63 M de lignes** : impossible à charger dans pandas sur un laptop. On utilise **DuckDB** : conversion unique en Parquet (~1 min, colonnée et compressée), puis on agrège en SQL et on ne rapatrie dans pandas que de petites tables pour tracer.

In [1]:
from pathlib import Path
import time
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mtick
from matplotlib.colors import LinearSegmentedColormap

pd.set_option("display.max_columns", 50, "display.width", 200, "display.float_format", lambda x: f"{x:,.3f}")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CSV = next((p for p in (ROOT / "data" / "dataset_sales_2026.csv", ROOT / "dataset_sales_2026.csv") if p.exists()), ROOT / "dataset_sales_2026.csv")
PARQUET = ROOT / "data" / "sales.parquet"

con = duckdb.connect()
con.execute("SET memory_limit='4GB'")
(ROOT / "data").mkdir(exist_ok=True)
con.execute(f"SET temp_directory='{ROOT / 'data' / '.duckdb_tmp'}'")


def q(sql: str) -> pd.DataFrame:
    """Run a SQL query and return a pandas DataFrame."""
    return con.execute(sql).fetchdf()


if not PARQUET.exists():
    t0 = time.time()
    con.execute(f"""
        COPY (SELECT * FROM read_csv('{CSV}', delim=';', header=true, columns={{
            'date_order':'DATE','date_invoice':'DATE','product_id':'BIGINT','client_id':'BIGINT',
            'sales_net':'DOUBLE','quantity':'DOUBLE','order_channel':'VARCHAR','branch_id':'BIGINT'}}))
        TO '{PARQUET}' (FORMAT parquet, COMPRESSION zstd)""")
    print(f"CSV -> Parquet en {time.time() - t0:.0f}s")

con.execute(f"CREATE OR REPLACE VIEW sales AS SELECT * FROM read_parquet('{PARQUET}')")
q("SELECT * FROM sales LIMIT 5")

,date_order,date_invoice,product_id,client_id,sales_net,quantity,order_channel,branch_id
0,2022-09-25,2022-09-25,2376333,188502,155.443,3.000,online,2732
1,2022-09-25,2022-09-25,2520527,835089,16.394,3.000,at the store,10279
2,2022-09-25,2022-09-25,2536756,1977896,365.766,21.000,online,6184
3,2022-09-25,2022-09-25,3028673,598816,8.599,201.000,at the store,4200
4,2022-09-25,2022-09-25,203377,2086861,"1,093.374",3.000,by phone,7051


### Style graphique
Palette catégorielle validée (ordre fixe : une couleur = une entité), rampe bleue pour les magnitudes, gris neutres pour la structure, orange pour « ce qui alerte ». Fond clair pour que les figures restent lisibles quel que soit le thème de l'éditeur.

In [3]:
C = dict(blue="#2a78d6", orange="#eb6834", aqua="#1baf7a", yellow="#eda100", magenta="#e87ba4",
         green="#008300", violet="#6250d6", red="#e34948")
INK, INK2, MUTED, GRID, AXIS, SURF = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
SEQ = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
CMAP = LinearSegmentedColormap.from_list("seq_blue", SEQ)
NEUTRAL = "#b9b8b0"

# Une couleur par canal, identique dans tout le notebook
CHANNEL_COLOR = {"by phone": C["orange"], "at the store": C["blue"], "online": C["aqua"],
                 "during the visit of a sales rep": C["yellow"], "other": C["magenta"]}
CHANNEL_LABEL = {"by phone": "Téléphone", "at the store": "En agence", "online": "En ligne",
                 "during the visit of a sales rep": "Visite commercial", "other": "Autre"}

plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "font.family": "sans-serif", "font.sans-serif": ["Helvetica Neue", "Arial", "DejaVu Sans"], "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlecolor": INK,
    "lines.linewidth": 2, "figure.dpi": 110, "legend.frameon": False, "legend.fontsize": 9,
    "axes.labelsize": 9.5, "xtick.labelsize": 9, "ytick.labelsize": 9,
})


def headline(fig, title, sub=None):
    """Action title (the takeaway) + optional muted subtitle, left-aligned above the figure."""
    h = fig.get_figheight()
    fig.text(0.005, 1 + (0.30 if sub else 0.12) / h, title, fontsize=13, fontweight="bold", color=INK, ha="left", va="bottom")
    if sub:
        fig.text(0.005, 1 + 0.08 / h, sub, fontsize=9.5, color=INK2, ha="left", va="bottom")


def fmt_m(x, _=None):
    return f"{x / 1e6:,.0f}"


def pct_axis(ax, axis="y", decimals=0):
    f = mtick.PercentFormatter(1.0, decimals=decimals)
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(f)


def label_end(ax, x, y, text, dx=6, **kw):
    """Direct label at the end of a line (secondary ink, never the series colour)."""
    ax.annotate(text, (x, y), xytext=(dx, 0), textcoords="offset points", va="center", color=INK2, fontsize=9, **kw)


def pct(x, d=1):
    return f"{100 * x:.{d}f} %"

### Table (client, jour d'achat)

Construite une fois à partir du Parquet (une passe sur les 63 M de lignes), comme dans `01_data_quality_and_activity_overview`.

In [8]:
# Daily client x day table (one pass over the 63M lines), then purchase days only.
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE cd AS
SELECT client_id, date_order AS d,
       sum(sales_net)                                   AS rev_net,
       sum(greatest(sales_net, 0))                      AS rev_pos,
       sum(least(sales_net, 0))                         AS rev_neg,
       count(*)                                         AS n_lines,
       count(DISTINCT product_id)                       AS n_prod,
       arg_max(branch_id, greatest(sales_net, 0))       AS branch_id,
       arg_max(order_channel, greatest(sales_net, 0))   AS channel,
       avg(least(greatest(date_diff('day', date_order, date_invoice), 0), 365)) AS lag_mean
FROM sales GROUP BY client_id, date_order""")
con.execute("CREATE OR REPLACE TABLE bd AS SELECT * FROM cd WHERE rev_pos > 0")
cnt = q("SELECT (SELECT count(*) FROM cd) AS client_days, (SELECT count(*) FROM bd) AS purchase_days").iloc[0]
print(f"{cnt['client_days']:,.0f} (client, jour) · dont {cnt['purchase_days']:,.0f} jours d'achat · construit en {time.time() - t0:.0f}s")

7,955,289 (client, jour) · dont 7,953,336 jours d'achat · construit en 16s


## 1. Cadrage du périmètre : churn total ou partiel, quels segments ?

**Question Q9 bis.** Avant de modéliser, combien de CA est **en jeu** selon la définition du churn et selon le segment ciblé ? Ce plafond de CA est la première moitié de l'arbitrage ; la seconde (fiabilité du modèle par périmètre) vient après. Aucun modèle dans cette section ; le test de modèle est dans `05_label_test_and_list_allocation`.

### Les définitions

On se place à une date de photo **T**, avec trois fenêtres consécutives de 180 jours : **A** = les 180 j avant T (26/03 → 21/09/2023), **B** = les 180 j après T (22/09/2023 → 19/03/2024), **C** = les 180 j suivants (20/03 → 15/09/2024). Pour chaque client, on additionne son CA et on compte ses jours d'achat dans chaque fenêtre ; rien d'autre n'est calculé à l'intérieur d'une fenêtre, sauf l'écart entre jours d'achat pour l'interruption.

| Terme | Définition | CA perdu retenu |
|---|---|---|
| **Jour d'achat** | Jour où le CA net du client (ventes moins retours du jour) est **> 0,01**. Un achat intégralement retourné le jour même ne compte pas. Le CA d'un jour d'achat est ce net | — |
| **Base** | Clients ayant au moins un jour d'achat dans A | — |
| **CA de référence** | CA de A × 2 (annualisé), valeur V par défaut du document de décisions | — |
| **Churn total** | Aucun jour d'achat dans B | CA de référence en entier |
| **Churn partiel** | Au moins un achat dans B, mais **CA(B) < 50 % de CA(A)** | (CA(A) − CA(B)) × 2 |
| **… dont durable** | Churn partiel **et** CA(C) < 50 % de CA(A) : le client ne rebondit pas | (CA(A) − CA(B)) × 2 |
| **Interruption** | Ni total ni partiel, mais un **silence ≥ H_i** dans B, clos par un achat dans B. H_i = k × écart p90 du client (calculé sur les 365 j avant T), borné entre 30 et 180 j, k = 3. Le silence est mesuré à partir de T au plus tôt. Demande 6 jours d'achat | CA attendu pendant le silence : CA(A)/180 × (silence − écart médian). **Perte ponctuelle, non annualisée** |
| **Contre-poids** | CA(B) > 150 % de CA(A) (hausse > 50 %) | CA **gagné**, pour mesurer la volatilité |

Total, partiel et interruption s'excluent (dans cet ordre de priorité) ; « durable » est un sous-ensemble du partiel. Le contre-poids n'est pas un churn : il montre à quel point le CA des clients bouge naturellement, donc à quel point un label « partiel » sera bruité. L'interruption est le « client qui disparaît 3 mois puis revient » : le churn total à 180 j ne le voit pas, et le partiel seulement s'il ne rattrape pas ensuite.

**Segments** (classes de rythme du document de décisions, calculées sur les 365 j avant T) : *historique court* (moins de 6 jours d'achat), puis, à partir de 6 achats, *rapide* (écart médian ≤ 7 j), *moyen* (8 à 30 j) et *lent* (> 30 j).

**Simplifications, à lever à l'étape modèle.** L'horizon est de 180 j pour tous les segments (le document initial prévoyait 30 à 180 j selon le rythme ; 180 j est retenu pour la vérité, voir `06_label_horizon_calibration`), ce qui rend les segments comparables en CA. Le seuil de 50 % est celui de la configuration (`drop_threshold`). Un retour fait un autre jour que l'achat n'est pas déduit du CA. Un silence encore ouvert à la fin de B n'est pas compté en interruption (il peut l'être en partiel).

**Une seule photo, qui exploite toutes les données.** T = 22/09/2023 : les 365 j d'historique avant T partent du début des données (22/09/2022) et les fenêtres B et C vont jusqu'au 15/09/2024, la fin des données étant le 22/09/2024. C'est la seule date qui laisse assez de recul des deux côtés pour mesurer aussi la durabilité d'une chute. Elle ne donne qu'une saison : les creux de mai et sept. 2023 sont dans l'historique ou dans A, celui de mai 2024 dans C.

In [34]:
# --- Paramètres du cadrage (modifiables)
T_SCOPE = "2023-09-22"                    # photo unique : 365 j d'historique avant, 360 j de recul après (toutes les données)
H_SC = 180                                # largeur des fenêtres A, B et C
DROP, RISE = 0.5, 0.5                     # churn partiel : CA(B) < 50 % de CA(A) ; contre-poids : CA(B) > 150 %
MIN_PURCH = 6                             # écarts d'achat fiables à partir de 6 jours d'achat sur 365 j
K_INT, H_MIN, H_MAX = 3.0, 30, 180        # interruption : silence >= clip(k x écart p90 du client, 30, 180 j)
SEGS = ["Rapide (écart méd. ≤ 7 j)", "Moyen (8–30 j)", "Lent (> 30 j)", "Historique court (< 6 achats)"]

# Jour d'achat = CA net du jour > 1 centime (un achat retourné le jour même n'est pas un achat ; le seuil évite que le
# bruit de virgule flottante d'un net de ±1e-13 fasse basculer le résultat d'une exécution à l'autre) ; `rev` = ce CA net
con.execute("CREATE OR REPLACE TABLE bn AS SELECT client_id, d, rev_net AS rev FROM cd WHERE rev_net > 0.01")
DATA_END = pd.Timestamp(q("SELECT max(d) AS m FROM bn").iloc[0, 0])


def scope_snapshot(T: str) -> pd.DataFrame:
    """One row per client of the base at T: CA of windows A (rev_a), B (rev_b), C (rev_c), purchase days in B,
    longest silence in B closed by a purchase (max_gap_b, measured from T at the earliest), rhythm and class."""
    df = q(f"""
    WITH hist AS (SELECT *, d - lag(d) OVER (PARTITION BY client_id ORDER BY d) AS gap
                  FROM bn WHERE d < DATE '{T}' AND d >= DATE '{T}' - 365),
    agg AS (SELECT client_id, count(*) AS n365, median(gap) AS med_gap, quantile_cont(gap, 0.9) AS p90_gap, max(d) AS last_d,
                   coalesce(sum(rev) FILTER (WHERE d >= DATE '{T}' - {H_SC}), 0) AS rev_a
            FROM hist GROUP BY 1),
    fut AS (SELECT client_id,
                   coalesce(sum(rev) FILTER (WHERE d >= DATE '{T}' AND d < DATE '{T}' + {H_SC}), 0) AS rev_b,
                   coalesce(sum(rev) FILTER (WHERE d >= DATE '{T}' + {H_SC}), 0) AS rev_c,
                   count(*) FILTER (WHERE d >= DATE '{T}' AND d < DATE '{T}' + {H_SC}) AS n_b,
                   max(gap_b) FILTER (WHERE d >= DATE '{T}' AND d < DATE '{T}' + {H_SC}) AS max_gap_b
            FROM (SELECT client_id, d, rev, d - greatest(lag(d) OVER (PARTITION BY client_id ORDER BY d), DATE '{T}') AS gap_b
                  FROM bn WHERE d >= DATE '{T}' - 365 AND d < DATE '{T}' + 2 * {H_SC}) GROUP BY 1)
    SELECT a.*, coalesce(f.rev_b, 0) AS rev_b, coalesce(f.rev_c, 0) AS rev_c, coalesce(f.n_b, 0) AS n_b, f.max_gap_b
    FROM agg a LEFT JOIN fut f USING (client_id)
    WHERE date_diff('day', a.last_d, DATE '{T}') <= {H_SC}""")
    df["seg"] = np.select([df.n365 < MIN_PURCH, df.med_gap <= 7, df.med_gap <= 30],
                          [SEGS[3], SEGS[0], SEGS[1]], default=SEGS[2])
    return df


def is_partial(df: pd.DataFrame) -> pd.Series:
    return (df.n_b > 0) & (df.rev_b < (1 - DROP) * df.rev_a)


def is_interruption(df: pd.DataFrame, k: float = K_INT) -> pd.Series:
    """Not total, not partial, but a silence >= clip(k x p90 gap, 30, 180 j) closed by a purchase in B."""
    h_i = (k * df.p90_gap).clip(H_MIN, H_MAX)               # NaN without enough history
    return (df.n_b > 0) & ~is_partial(df) & (df.max_gap_b >= h_i) & (df.n365 >= MIN_PURCH)


def silence_loss(df: pd.DataFrame) -> pd.Series:
    """CA expected during the silence (one-off, not annualised)."""
    return df.rev_a / H_SC * (df.max_gap_b - df.med_gap).clip(lower=0)


def scope_table(T: str) -> pd.DataFrame:
    """Sizing table: for each segment x churn type, clients, CA of those clients, CA lost (M)."""
    df = scope_snapshot(T)
    ann = 2 * 180 / H_SC                                    # annualisation factor of a window of H_SC days
    assert pd.Timestamp(T) + pd.Timedelta(days=2 * H_SC) <= DATA_END, "pas assez de recul pour la fenêtre C"
    df["ref"] = ann * df.rev_a
    df["delta"] = ann * (df.rev_a - df.rev_b)               # run-rate CA lost between A and B
    total = df.n_b == 0
    partial, interrupt = is_partial(df), is_interruption(df)
    types = {"Base du segment": (pd.Series(True, index=df.index), None),
             "Churn total": (total, df.ref),
             "Churn partiel (brut)": (partial, df.delta),
             "… dont durable": (partial & (df.rev_c < (1 - DROP) * df.rev_a), df.delta),
             "Interruption (hors partiel, perte ponctuelle)": (interrupt, silence_loss(df)),
             "Contre-poids : hausse > 50 %": ((df.n_b > 0) & (df.rev_b > (1 + RISE) * df.rev_a), -df.delta)}
    n_all, ca_all = len(df), df.ref.sum()
    rows = []
    for seg, g in [("Tous les segments", df)] + [(s, df[df.seg == s]) for s in SEGS]:
        for name, (mask, lost) in types.items():
            m = mask.loc[g.index]
            rows.append({"Segment": seg, "Type": name, "T": T, "n": m.sum(), "pct_n": m.sum() / n_all,
                         "ca": g.ref[m].sum() / 1e6, "pct_ca": g.ref[m].sum() / ca_all,
                         "lost": np.nan if lost is None else lost.loc[g.index][m].sum() / 1e6,
                         "pct_lost": np.nan if lost is None else lost.loc[g.index][m].sum() / ca_all})
    return pd.DataFrame(rows)


def show_scope(t: pd.DataFrame) -> pd.DataFrame:
    f = pd.DataFrame({"Clients": t.n.map("{:,.0f}".format), "% des clients": t.pct_n.map("{:.1%}".format),
                      "CA de ces clients (M)": t.ca.map("{:,.0f}".format), "% du CA total": t.pct_ca.map("{:.1%}".format),
                      "CA perdu ou gagné (M)": t.lost.map(lambda v: "" if pd.isna(v) else f"{v:,.0f}"),
                      "% du CA total ": t.pct_lost.map(lambda v: "" if pd.isna(v) else f"{v:.1%}")})
    return f.set_index([t.Segment, t.Type])


scope = scope_table(T_SCOPE)
base_n = scope.query("Segment == 'Tous les segments' and Type == 'Base du segment'").iloc[0]
print(f"Photo du {T_SCOPE} : base = {base_n.n:,.0f} clients actifs sur 180 j, CA de référence annualisé = {base_n.ca:,.0f} M "
      f"(dénominateur des « % du CA total »). Colonne de droite : CA perdu pour les churns (annualisé, sauf l'interruption : perte ponctuelle), "
      f"CA gagné pour le contre-poids.")
show_scope(scope)

Photo du 2023-09-22 : base = 118,009 clients actifs sur 180 j, CA de référence annualisé = 4,410 M (dénominateur des « % du CA total »). Colonne de droite : CA perdu pour les churns (annualisé, sauf l'interruption : perte ponctuelle), CA gagné pour le contre-poids.


Clients % des clients CA de ces clients (M) % du CA total CA perdu ou gagné (M) % du CA total 
Segment                       Type                                                                                                                                         
Tous les segments             Base du segment                                118,009        100.0%                 4,410        100.0%                                     
                              Churn total                                     22,095         18.7%                   125          2.8%                   125           2.8%
                              Churn partiel (brut)                            23,114         19.6%                   705         16.0%                   509          11.5%
                              … dont durable                                  14,509         12.3%                   474         10.7%                   357           8.1%
                              Interruption (hors partiel, perte ponctuelle)    2,853          2.4%                    78          1.8%                    11           0.2%
                              Contre-poids : hausse > 50 %                    31,509         26.7%                   604         13.7%                   826          18.7%
Rapide (écart méd. ≤ 7 j)     Base du segment                                 42,288         35.8%                 3,892         88.3%                                     
                              Churn total                                      2,375          2.0%                    57          1.3%                    57           1.3%
                              Churn partiel (brut)                             7,753          6.6%                   531         12.0%                   374           8.5%
                              … dont durable                                   5,012          4.2%                   353          8.0%                   262           5.9%
                              Interruption (hors partiel, perte ponctuelle)    2,292          1.9%                    73          1.6%                     9           0.2%
                              Contre-poids : hausse > 50 %                     9,841          8.3%                   506         11.5%                   576          13.1%
Moyen (8–30 j)                Base du segment                                 33,194         28.1%                   373          8.5%                                     
                              Churn total                                      3,037          2.6%                    20          0.4%                    20           0.4%
                              Churn partiel (brut)                             8,503          7.2%                   128          2.9%                    97           2.2%
                              … dont durable                                   5,115          4.3%                    86          1.9%                    67           1.5%
                              Interruption (hors partiel, perte ponctuelle)      556          0.5%                     5          0.1%                     1           0.0%
                              Contre-poids : hausse > 50 %                    10,073          8.5%                    77          1.7%                   155           3.5%
Lent (> 30 j)                 Base du segment                                  6,211          5.3%                    30          0.7%                                     
                              Churn total                                        842          0.7%                     3          0.1%                     3           0.1%
                              Churn partiel (brut)                             1,517          1.3%                    11          0.2%                     8           0.2%
                              … dont durable                                     893          0.8%                     7 

In [35]:
# Sensibilité de l'interruption au multiplicateur k (seuil = k x écart p90 du client, borné entre 30 et 180 j)
snap = scope_snapshot(T_SCOPE)
snap["ref"] = 2 * snap.rev_a
sens = []
for k in (1.5, 2, 3, 5):
    m = is_interruption(snap, k)
    r_ = m & snap.seg.eq(SEGS[0])
    sens.append({"k": k, "Clients": f"{m.sum():,.0f}", "dont rapides": f"{r_.sum():,.0f}",
                 "CA de ces clients (M)": f"{snap.ref[m].sum() / 1e6:,.0f}", "% du CA total": f"{snap.ref[m].sum() / snap.ref.sum():.1%}",
                 "Perte ponctuelle (M)": f"{silence_loss(snap)[m].sum() / 1e6:,.0f}"})
pd.DataFrame(sens).set_index("k")

,Clients,dont rapides,CA de ces clients (M),% du CA total,Perte ponctuelle (M)
k,,,,,
1.500,"12,948","6,854",238,5.4%,29
2.000,"7,704","5,016",166,3.8%,21
3.000,"2,853","2,292",78,1.8%,11
5.000,711,655,26,0.6%,4


**Lecture** (plafonds de CA, pas encore des impacts : ni prédictibilité, ni rétention, ni décote ρ). Pourcentages en part du CA de référence de la base ; le churn est mesuré sur une fenêtre de 180 j, donc ≈ 2 fois plus sur un an, de même ordre que les 3,4 à 4,8 % mesurés dans `02_clients_rhythm_and_churn_definition`.

- **Le CA est dans les clients rapides.** Ils font 36 % des clients mais **88 % du CA**. Les clients lents (5 % des clients) pèsent **0,7 % du CA** et leur churn total 0,1 % : l'hypothèse « occasionnels à longue fréquence » ne vaut pas le coup, quel que soit le modèle.
- **Churn total : un gisement de 125 M (2,8 % du CA).** Il se répartit entre rapides (46 %), historique court (36 %) et moyens (16 %). Les clients à historique court sont nombreux (13 % de la base en churn total) mais sans variable de rythme : une baseline seule, au mieux.
- **Churn partiel : un plafond plus haut, mais bruité.** Les chutes de plus de 50 % représentent 509 M (11,5 % du CA), dont **357 M durables (8,1 %)**, presque 3 fois le churn total, et 73 % de ce montant vient des seuls clients rapides. Mais le **contre-poids est plus grand encore** : 826 M gagnés par des hausses de plus de 50 %. Chez les rapides, 23 % des clients montent de plus de 50 % et 18 % baissent de plus de 50 % : le CA oscille fortement d'un semestre à l'autre (chantiers, commandes groupées). Le label partiel sera donc plus difficile à prédire que le total, et son CA « récupérable » est plus incertain.
- **Interruption : réelle mais petite.** Le client rapide qui s'arrête plusieurs semaines puis revient existe : 2 853 clients (dont 80 % de rapides) ont un silence de plus de 3 fois leur écart p90 sans être déjà classés en partiel. Mais la perte est **ponctuelle d'environ 11 M** (0,2 % du CA), contre 125 M par an pour le churn total. Le résultat dépend de k, sans changer la conclusion : à k = 1,5, 12 948 clients et 29 M de perte ponctuelle. Ce type n'est donc pas une cible prioritaire ; il vaut surtout comme variable du modèle (un silence inhabituel précède souvent une chute).
- **Retours.** Compter comme achat un jour à CA net > 0,01 (au lieu d'une simple ligne positive) retire 1 647 clients de la base (119 656 → 118 009) et 6 % du CA de référence (4 687 → 4 410 M), car le CA est maintenant net des retours du jour. Le churn total reste à 2,8 % du CA.
- **Contrainte de label.** Le label « durable » demande 360 j de recul après la photo : sur ces données, une seule photo de ce type est possible (celle-ci), contre 6 à 7 pour le churn total. L'entraînement d'un modèle durable aurait donc très peu de photos.
- **Limite.** Une seule photo, donc une seule saison. Les silences encore ouverts à la fin de B ne sont pas comptés en interruption (ils le sont en partiel s'ils font chuter le CA de plus de 50 %).

**Pistes pour l'étape suivante (fiabilité).** Rapide et moyen ensemble portent 62 % du CA du churn total, et rapide seul 73 % du churn partiel durable. Les options à comparer sont donc : total tous segments, total rapide + moyen, partiel durable rapide.

In [36]:
# Horizon de perte : parmi les clients sans achat pendant H jours après T, combien rachètent quand même avant T + 180 j ?
HORIZONS = [45, 60, 90, 120, 150, 180]
nxt = q(f"SELECT client_id, date_diff('day', DATE '{T_SCOPE}', min(d)) AS dnext FROM bn WHERE d >= DATE '{T_SCOPE}' GROUP BY 1")
hz = scope_snapshot(T_SCOPE).merge(nxt, on="client_id", how="left")
hz["ref"] = 2 * hz.rev_a
ca_all = hz.ref.sum()
rows = []
for seg in [SEGS[0], SEGS[1], "Rapide + moyen"]:
    g = hz[hz.seg.isin(SEGS[:2])] if seg == "Rapide + moyen" else hz[hz.seg == seg]
    for H in HORIZONS:
        out = g.dnext.isna() | (g.dnext >= H)                 # aucun jour d'achat dans [T, T + H)
        back = out & g.dnext.notna() & (g.dnext < H_SC)       # ... mais un achat avant T + 180 j
        rows.append({"Segment": seg, "Horizon H (j)": H, "Clients sans achat": f"{out.sum():,.0f}",
                     "% du segment": f"{out.sum() / len(g):.1%}", "CA de ces clients (M)": f"{g.ref[out].sum() / 1e6:,.0f}",
                     "% du CA total": f"{g.ref[out].sum() / ca_all:.1%}",
                     "Reviennent avant 180 j": f"{back.sum() / out.sum():.0%}",
                     "dont CA qui revient (M)": f"{g.ref[back].sum() / 1e6:,.0f}",
                     "CA des départs définitifs (M)": f"{g.ref[out & ~back].sum() / 1e6:,.0f}"})
pd.DataFrame(rows).set_index(["Segment", "Horizon H (j)"])

Clients sans achat % du segment CA de ces clients (M) % du CA total Reviennent avant 180 j dont CA qui revient (M) CA des départs définitifs (M)
Segment                   Horizon H (j)                                                                                                                                                 
Rapide (écart méd. ≤ 7 j) 45                         4,599        10.9%                    92          2.1%                    48%                      36                            57
                          60                         3,873         9.2%                    79          1.8%                    39%                      23                            57
                          90                         3,103         7.3%                    68          1.5%                    23%                      11                            57
                          120                        2,760         6.5%                    62          1.4%                    14%                       6                            57
                          150                        2,540         6.0%                    59          1.3%                     6%                       2                            57
                          180                        2,375         5.6%                    57          1.3%                     0%                       0                            57
Moyen (8–30 j)            45                         9,866        29.7%                    73          1.7%                    69%                      54                            20
                          60                         7,685        23.2%                    53          1.2%                    60%                      33                            20
                          90                         5,289        15.9%                    34          0.8%                    43%                      15                            20
                          120                        4,184        12.6%                    26          0.6%                    27%                       7                            20
                          150                        3,541        10.7%                    22          0.5%                    14%                       3                            20
                          180                        3,037         9.1%                    20          0.4%                     0%                       0                            20
Rapide + moyen            45                        14,465        19.2%                   166          3.8%                    63%                      90                            76
                          60                        11,558        15.3%                   132          3.0%                    53%                      56                            76
                          90                         8,392        11.1%                   102          2.3%                    36%                      25                            76
                          120                        6,944         9.2%                    89          2.0%                    22%                      12                            76
                          150                        6,081         8.1%                    81          1.8%                    11%                       5                            76
                          180                        5,412         7.2%                    76          1.7%                     0%                       0                            76

**Lecture du tableau des horizons** (une photo, T = 22/09/2023). Chaque ligne répond à : « si on déclare perdu un client sans achat pendant H jours, quelle part reviendra quand même avant T + 180 j ? ». Les « départs définitifs » sont les clients sans achat sur les 180 j : ils sont les mêmes quel que soit H (57 M rapides, 20 M moyens) ; tout le reste du CA d'une ligne est du CA qui revient.

- **Plus H est court, plus la liste se remplit de clients qui reviendraient seuls.** Rapides : 48 % de retours à 45 j, 39 % à 60 j, 23 % à 90 j, 14 % à 120 j, 6 % à 150 j.
- **Les moyens demandent un horizon plus long, pas plus court.** À 60 j (la valeur de départ du document de décisions), 60 % des « perdus » reviennent ; il faut 120 j pour redescendre à 27 %. Le document de décisions a donc l'ordre inverse de ce que montrent les données sur ce point : 60 j pour les moyens est trop court.
- **À tolérance égale (≈ 25 % de retours), ≈ 90 j pour les rapides et ≈ 120 j pour les moyens.** L'horizon d'un moyen est environ 1,3 fois celui d'un rapide, ce qui va dans le sens d'un label relatif au rythme du client plutôt que de deux horizons fixes.
- **Ce que coûte un horizon court, en CA.** À 45 j chez les rapides, 92 M sont déclarés perdus pour 57 M de départs réels : 36 M, soit 39 % du CA signalé, reviennent seuls. À 90 j, c'est 11 M sur 68 M (16 %).
- **Ce que rapporte un horizon court : du temps pour agir.** Il n'est pas mesuré ici. Il se règle à l'étape modèle, par les variables d'alerte (chute de fréquence, récence relative), plutôt que par le label.

*Limites.* Les retours sont comptés jusqu'à T + 180 j seulement : un client qui revient après est compté « parti » à 180 j, donc la part de faux départs est légèrement sous-estimée. Une seule photo, donc une seule saison.

**Correction du 24/11 : le tableau des horizons ci-dessus est circulaire.** Sa fenêtre de retour s'arrête à T + 180 j, donc la colonne 180 j vaut 0 % par construction et la fenêtre rétrécit quand H augmente : il ne peut pas montrer que 180 j est le bon seuil. Recalcul ci-dessous avec une **fenêtre de retour fixe de 90 j** (photos mensuelles poolées du 22/09/2023 au 22/06/2024, seules celles dont la fenêtre H + 90 j tient dans les données).

**Lecture.** (i) Les horizons courts sont exclus : à 60 j, 37 % des rapides et 57 % des moyens « perdus » rachètent dans les 90 j suivants. (ii) Pas de plateau : à 180 j, 13 % des rapides et 27 % des moyens (8 % et 24 % de leur CA) reviennent encore ; à 240 j, 10 % et 21 %. (iii) À horizon égal, les moyens reviennent deux fois plus que les rapides. **180 j est donc une convention, pas un seuil imposé par les données** ; le churn total est « aucun achat en 6 mois », avec ≈ 12 % de son CA (rapides + moyens) qui revient dans les 3 mois suivants. Les photos se chevauchent (mêmes clients d'un mois à l'autre) : les effectifs ne sont pas indépendants.

In [ ]:
# Horizon de perte, recalcul (24/11) : fenêtre de retour FIXE de K_RET jours, photos mensuelles poolées.
# Cohorte(H) = clients réguliers sans achat dans [T, T + H) ; retour = achat dans [T + H, T + H + K_RET).
K_RET = 90
HORIZONS = [30, 60, 90, 120, 180, 240]
T_POOL = [str(t.date()) for t in pd.date_range("2023-09-22", "2024-06-22", freq=pd.DateOffset(months=1))]
nxt_all = lambda T: q(f"SELECT client_id, date_diff('day', DATE '{T}', min(d)) AS dnext FROM bn WHERE d >= DATE '{T}' GROUP BY 1")
frames = []
for T in T_POOL:
    f = scope_snapshot(T).merge(nxt_all(T), on="client_id", how="left")
    f["T"], f["ca"] = pd.Timestamp(T), 2 * f.rev_a
    frames.append(f[f.seg.isin(SEGS[:2])])
reg = pd.concat(frames, ignore_index=True)

rows = []
for H in HORIZONS:
    g = reg[reg["T"] + pd.Timedelta(days=H + K_RET) <= DATA_END]          # la fenêtre de retour doit tenir dans les données
    out = g[g.dnext.isna() | (g.dnext >= H)]
    for seg in SEGS[:2] + ["Rapide + moyen"]:
        s = out if seg == "Rapide + moyen" else out[out.seg == seg]
        back = s.dnext.notna() & (s.dnext < H + K_RET)
        rows.append({"Segment": seg, "H (j)": H, "Photos": g["T"].nunique(), "Clients sans achat": len(s),
                     "% qui reviennent": f"{back.mean():.0%}", "% du CA qui revient": f"{(s.ca * back).sum() / s.ca.sum():.0%}"})
pd.DataFrame(rows).set_index(["Segment", "H (j)"])

# Même mesure à H = 180 j (churn total) avec un retour sur 180 j : une seule photo tient dans les données
g = reg[reg["T"] + pd.Timedelta(days=360) <= DATA_END]
out = g[g.dnext.isna() | (g.dnext >= 180)]
for seg in SEGS[:2] + ["Rapide + moyen"]:
    s = out if seg == "Rapide + moyen" else out[out.seg == seg]
    back = s.dnext.notna() & (s.dnext < 360)
    print(f"Churn total, retour sous 180 j · {seg:<26} {len(s):>6,} clients · {back.mean():.0%} reviennent · {(s.ca * back).sum() / s.ca.sum():.0%} du CA")

### Saisonnalité : à quel point la date de la photo change-t-elle les labels ?

Jusqu'ici une seule photo (T = 22/09/2023), donc une seule saison. Deux contrôles : (1) le CA par jour selon le mois, pour voir l'ampleur du motif saisonnier ; (2) les taux de churn total, partiel et de hausse sur **13 photos mensuelles** (T du 22/03/2023 au 22/03/2024 : A a besoin de 180 j d'historique avant T, B de 180 j de données après). Les définitions sont celles de la section ci-dessus ; « réguliers » est approché ici par au moins 15 jours d'achat dans A.

In [37]:
# Saisonnalité 1/2 : CA par jour selon le mois (indice, 100 = moyenne des 23 mois complets)
sm = q("""SELECT date_trunc('month', d) AS m, sum(rev_net) AS ca FROM cd
          WHERE d >= DATE '2022-10-01' AND d < DATE '2024-09-01' GROUP BY 1 ORDER BY 1""")
sm["ca_j"] = sm.ca / sm.m.dt.days_in_month
sm["indice"] = 100 * sm.ca_j / sm.ca_j.mean()
MOIS = ["janv.", "févr.", "mars", "avr.", "mai", "juin", "juil.", "août", "sept.", "oct.", "nov.", "déc."]
sm["mois"], sm["année"] = sm.m.dt.month, sm.m.dt.year
si = sm.pivot(index="mois", columns="année", values="indice")
si.index = [MOIS[i - 1] for i in si.index]
same = lambda a, b: sm[(sm.m >= a) & (sm.m <= b)].ca.sum()      # mêmes 11 mois, deux années de suite
print(f"Même période d'une année sur l'autre (oct. à août) : {same('2023-10-01', '2024-08-01') / same('2022-10-01', '2023-08-01') - 1:+.1%}")
si.map(lambda v: "" if pd.isna(v) else f"{v:.0f}")

Même période d'une année sur l'autre (oct. à août) : +4.7%


année,2022,2023,2024
janv.,,97,92
févr.,,102,95
mars,,108,113
avr.,,82,98
mai,,71,75
juin,,102,106
juil.,,106,117
août,,118,123
sept.,,76,
oct.,105,110,


## 2. Le périmètre des réguliers sur 180 jours

Le tableau de segments ci-dessus classe le rythme sur **365 jours**. Le périmètre qu'utilisent les modèles est défini sur **180 jours** : au moins 6 jours d'achat sur les 180 jours avant la photo et un écart médian ≤ 30 jours. Même photo (22/09/2023), même base de 118 009 clients : combien pèsent les segments avec cette définition ?

In [ ]:
T = "2023-09-22"
df = q(f"""
WITH hist AS (SELECT client_id, d, rev, d - lag(d) OVER (PARTITION BY client_id ORDER BY d) AS gap
              FROM bn WHERE d < DATE '{T}' AND d >= DATE '{T}' - 180),
agg AS (SELECT client_id, count(*) AS n, median(gap) AS med, sum(rev) AS rev_a FROM hist GROUP BY 1),
fut AS (SELECT client_id, min(d) - DATE '{T}' AS ng FROM bn WHERE d >= DATE '{T}' AND d < DATE '{T}' + 180 GROUP BY 1)
SELECT a.*, f.ng FROM agg a LEFT JOIN fut f USING (client_id)""")
df["V"] = 2 * df.rev_a
df["gone"] = df.ng.isna()
df["Segment (rythme sur 180 j)"] = np.select([df.n < 6, df.med <= 7, df.med <= 30],
                                             ["Historique court (< 6 jours d'achat)", "Rapide (écart médian ≤ 7 j)", "Moyen (8 à 30 j)"],
                                             default="Lent (> 30 j)")
tot_v, tot_loss = df.V.sum(), (df.V * df.gone).sum()
rows = []
for s_, g in df.groupby("Segment (rythme sur 180 j)"):
    rows.append({"Segment (rythme sur 180 j)": s_, "Clients (%)": 100 * len(g) / len(df), "CA (%)": 100 * g.V.sum() / tot_v,
                 "Churn total (M€)": (g.V * g.gone).sum() / 1e6, "Churn total (%)": 100 * (g.V * g.gone).sum() / tot_loss})
reg = df[df["Segment (rythme sur 180 j)"].isin(["Rapide (écart médian ≤ 7 j)", "Moyen (8 à 30 j)"])]
rows.append({"Segment (rythme sur 180 j)": "Réguliers (rapides + moyens) : périmètre du modèle", "Clients (%)": 100 * len(reg) / len(df),
             "CA (%)": 100 * reg.V.sum() / tot_v, "Churn total (M€)": (reg.V * reg.gone).sum() / 1e6,
             "Churn total (%)": 100 * (reg.V * reg.gone).sum() / tot_loss})
print(f"base : {len(df):,} clients, CA annualisé {tot_v / 1e6:,.0f} M€, churn total {tot_loss / 1e6:,.0f} M€ ({100 * tot_loss / tot_v:.1f} % du CA) ; réguliers : {len(reg):,} clients")
pd.DataFrame(rows).set_index("Segment (rythme sur 180 j)")

base : 118,009 clients, CA annualisé 4,410 M€, churn total 125 M€ (2.8 % du CA) ; réguliers : 60,312 clients


,Clients (%),CA (%),Churn total (M€),Churn total (%)
Segment (rythme sur 180 j),,,,
Historique court (< 6 jours d'achat),48.581,4.812,59.141,47.477
Lent (> 30 j),0.311,0.063,0.123,0.099
Moyen (8 à 30 j),16.894,6.873,11.640,9.345
Rapide (écart médian ≤ 7 j),34.214,88.251,53.662,43.079
Réguliers (rapides + moyens) : périmètre du modèle,51.108,95.125,65.302,52.424


**Lecture** (périmètre à 180 jours). Les réguliers (rapides + moyens) sont **51 % des clients, 95 % du CA et 52 % du CA du churn total** (125 M€ au total), contre 64 %, 97 % et 62 % avec le rythme sur 365 jours : définie sur 180 jours, la règle « au moins 6 achats » écarte davantage de clients. Les rapides pèsent 88 % du CA et 43 % du churn total ; les lents ne comptent pas. **L'historique court représente 47,5 % du CA du churn total (59 M€)** mais seulement 4,8 % du CA : c'est un gisement de petits clients, sans variables de rythme (au mieux une baseline). Le périmètre du modèle couvre donc environ la moitié du CA du churn total, mais 95 % du CA à protéger.